# SignSight — 1. EDA SupSign trước khi train

Dùng **CPU runtime**; không cần GPU, Torch hay weights. Dán link ZIP SupSign vào SOURCE_URL.
Chạy lần lượt tới cuối để tải **ZIP báo cáo EDA** rồi gửi file đó để rà soát trước khi train.
Báo cáo gồm audit nguồn 107 lớp và thống kê/bbox của subset CAR35 dành cho ô tô.
Lọc 35 lớp là cấu hình hiện tại, có thể điều chỉnh sau khi đọc báo cáo; chưa phải mapping QCVN đã duyệt.
Cache giữ ảnh/nhãn/split đã kiểm tra cho notebook train. Lần đầu phải đọc/hash toàn bộ ZIP nên có thể lâu.
Notebook chạy độc lập, không clone repo; lưu notebook không lưu dataset.
Nhóm filename/dHash chỉ là heuristic; test chưa chứng minh hiệu quả thực địa độc lập.

In [ ]:
import sys, subprocess, os
os.environ['YOLO_AUTOINSTALL'] = 'false'
os.environ['WANDB_MODE'] = 'disabled'
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'Pillow>=10,<13', 'matplotlib>=3.8,<4', 'PyYAML>=6,<7', 'tqdm>=4.66,<5', 'gdown==5.2.0'])
print('EDA dùng CPU; không giữ GPU trong lúc xử lý dữ liệu.')

## 1. Cấu hình

SAVE_TO_DRIVE=True giữ cache/kết quả trên Drive của tài khoản hiện tại.
False chỉ lưu trong runtime; tải kết quả trước khi ngắt. Link gdown phải là **file ZIP** cho phép tải qua link,
không phải folder; quota và quyền Drive vẫn áp dụng. Không đưa link riêng vào GitHub public.
Giữ DATASET_VERSION giống nhau ở hai notebook. Đổi version khi thay dữ liệu/lớp/grouping.

GROUPS_CSV tùy chọn có cột path,group cho mọi ảnh/alias hợp lệ. Không tự coi tên frame là chuyến xe.

In [ ]:
from pathlib import Path
WORK = Path('/content/signsight_supsign')
SAVE_TO_DRIVE = True
DRIVE_OUTPUT_ROOT = Path('/content/drive/MyDrive/SignSight/supsign_training')
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PERSIST = DRIVE_OUTPUT_ROOT
else:
    PERSIST = WORK / 'outputs'
DATASET_VERSION = 'supsign107-car35-v3'
PREPARED_CACHE_URL = ''
PREPARED_CACHE_SHA256 = ''
IMAGE_SIZE = 640
assert DATASET_VERSION and Path(DATASET_VERSION).name == DATASET_VERSION and DATASET_VERSION not in ('.', '..')
WORK.mkdir(parents=True, exist_ok=True)
PERSIST.mkdir(parents=True, exist_ok=True)
CACHE_ZIP = PERSIST / 'datasets' / (DATASET_VERSION + '.zip')
DATASET_ROOT = WORK / DATASET_VERSION
os.environ['YOLO_CONFIG_DIR'] = str(WORK / '.ultralytics')
print('Output:', PERSIST)
print('Cache:', CACHE_ZIP)

SOURCE_URL = ''  # Link file ZIP SupSign gốc hoặc file ID
SOURCE_SHA256 = ''  # Tùy chọn checksum ZIP gốc
SOURCE_ZIP = WORK / 'downloads' / DATASET_VERSION / 'source_supsign.zip'
GROUPS_CSV = None
SEED = 42
FRAME_BLOCK_SIZE = 100
NEAR_HASH_DISTANCE = 4

In [ ]:
# Code đã kiểm thử, nhúng trong notebook để chạy độc lập.
import json, shutil, csv, zipfile
TRAINING_FILES = {'scripts/dataset_eda.py': '"""Compatibility import; implementation lives in training.data.audit."""\nimport sys\nimport training.data.audit as _module\nif __name__ == "__main__" and hasattr(_module,"main"):\n    _module.main()\nelse:\n    sys.modules[__name__] = _module\n', 'scripts/supsign_training.py': '"""Compatibility import; implementation lives in training.data.supsign."""\nimport sys\nimport training.data.supsign as _module\nif __name__ == "__main__" and hasattr(_module,"main"):\n    _module.main()\nelse:\n    sys.modules[__name__] = _module\n', 'scripts/colab_downloads.py': '"""Compatibility import; implementation lives in training.data.downloads."""\nimport sys\nimport training.data.downloads as _module\nif __name__ == "__main__" and hasattr(_module,"main"):\n    _module.main()\nelse:\n    sys.modules[__name__] = _module\n', 'training/__init__.py': '"""Offline datasets, experiments and notebook tooling; never imported by the API."""\n', 'training/compatibility.py': '"""Explicit cache provenance checks; content identity is separate from file checksum."""\nimport hashlib\nimport json\nfrom pathlib import Path\n\nLEGACY_SIGNATURES = {\n (\'42132e8eb337a7d8c27ef35eb5b81688f879d01afaccf593fa88265c435f4eba\',\n  \'f89bb8e458fc951cb997f2de1e57497eeaf61f97a5824e179b66b33bde3e1e03\'),\n (\'ca8ef48fb483c142085aae0dc6ff9bb570074eac968c686ad4ec2ae4672bbf51\',\n  \'1893e87f0b5fff1719b1590a1bbbb511c1dd40714cf8e3ff4c7bf760eb5b91b1\'),\n}\n\ndef implementation_digest():\n    root=Path(__file__).parent\n    names=[\'compatibility.py\',\'data/common.py\',\'data/supsign.py\',\'data/cache.py\',\n           \'experiments/train.py\',\'experiments/evaluate.py\']\n    content={name:(root/name).read_text(encoding=\'utf-8\') for name in names}\n    return hashlib.sha256(json.dumps(content,sort_keys=True).encode()).hexdigest()\n\ndef require_compatible_recipe(saved, expected=None):\n    from training.data.common import recipe\n    try:\n        current=recipe(saved[\'classes\'],saved[\'seed\'],saved[\'block_size\'],saved[\'near_threshold\'])\n        current[\'groups_sha256\']=saved[\'groups_sha256\']\n        legacy = saved[\'version\']==1 and (saved[\'implementation_sha256\'],saved[\'reader_sha256\']) in LEGACY_SIGNATURES\n        if legacy:\n            comparable={k:v for k,v in saved.items() if k not in (\'version\',\'implementation_sha256\',\'reader_sha256\')}\n            if comparable!={k:v for k,v in current.items() if k not in (\'version\',\'implementation_sha256\',\'reader_sha256\')}:\n                raise ValueError(\'Legacy cache recipe parameters changed\')\n        elif saved!=current:\n            raise ValueError(\'Unsupported cache recipe/code; use a new dataset version\')\n        if expected is not None:\n            keys=(\'classes\',\'seed\',\'block_size\',\'near_threshold\',\'groups_sha256\')\n            if any(saved.get(k)!=expected.get(k) for k in keys):\n                raise ValueError(\'Cache recipe changed; choose a new cache version\')\n    except (KeyError,TypeError) as error:\n        raise ValueError(\'Invalid cache recipe\') from error\n\ndef dataset_content_fingerprint(manifest):\n    content={k:manifest[k] for k in (\'names\',\'source_zip_sha256\',\'recipe\')}\n    content[\'images\']=sorted(({k:r[k] for k in (\'sha256\',\'label_sha256\',\'group\',\'split\')}\n                              for r in manifest[\'images\']),key=lambda r:r[\'sha256\'])\n    return hashlib.sha256(json.dumps(content,sort_keys=True,separators=(\',\',\':\'),allow_nan=False).encode()).hexdigest()\n', 'training/data/__init__.py': '', 'training/data/audit.py': '"""Read-only detection EDA, including files inside ZIPs. No training or remapping."""\nfrom collections import Counter, defaultdict\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path, PurePosixPath\nimport random\nimport xml.etree.ElementTree as ET\nimport zipfile\n\nfrom PIL import Image\nimport yaml\n\nIMAGE_EXTENSIONS = {\'.jpg\', \'.jpeg\', \'.png\', \'.bmp\', \'.webp\', \'.tif\', \'.tiff\'}\n\n\nclass DatasetReader:\n    def __init__(self, root):\n        self.root = Path(root).resolve()\n        if self.root.is_file() and self.root.suffix.lower() == \'.zip\':\n            paths, base = [self.root], self.root.parent\n        elif self.root.is_dir():\n            paths, base = sorted(self.root.rglob(\'*\')), self.root\n        else:\n            raise FileNotFoundError(f\'Dataset directory or ZIP not found: {self.root}\')\n        self.entries, self.zips, self.warnings = {}, {}, []\n        for path in paths:\n            if not path.is_file() or path.is_symlink():\n                continue\n            name = path.relative_to(base).as_posix()\n            self.entries[name] = (path, None, path.stat().st_size)\n            if path.suffix.lower() == \'.zip\':\n                try:\n                    archive = zipfile.ZipFile(path)\n                    self.zips[name] = archive\n                    for item in archive.infolist():\n                        if not item.is_dir():\n                            self.entries[name + \'!/\' + item.filename] = (archive, item, item.file_size)\n                            if PurePosixPath(item.filename).suffix.lower() in {\'.zip\', \'.rar\', \'.7z\', \'.gz\', \'.tar\'}:\n                                self.warnings.append(f\'Nested archive not expanded: {name}!/{item.filename}\')\n                except (OSError, zipfile.BadZipFile) as error:\n                    self.warnings.append(f\'Cannot open archive {name}: {type(error).__name__}\')\n\n    def open(self, name):\n        source, member, _ = self.entries[name]\n        return source.open(member) if member is not None else source.open(\'rb\')\n\n    def read(self, name, max_bytes=256 * 1024 * 1024):\n        if self.entries[name][2] > max_bytes:\n            raise ValueError(f\'Annotation exceeds {max_bytes // 1024**2} MiB read limit\')\n        with self.open(name) as stream:\n            value = stream.read(max_bytes + 1)\n        if len(value) > max_bytes:\n            raise ValueError(\'Annotation read limit exceeded\')\n        return value\n\n    def close(self):\n        for archive in self.zips.values():\n            archive.close()\n\n    def __enter__(self):\n        return self\n\n    def __exit__(self, *_):\n        self.close()\n\n\ndef split_name(name):\n    mapping = {\'train\': \'train\', \'training\': \'train\', \'val\': \'val\', \'valid\': \'val\',\n               \'validation\': \'val\', \'test\': \'test\', \'testing\': \'test\'}\n    found = {mapping[part.casefold()] for part in PurePosixPath(name).parts[:-1] if part.casefold() in mapping}\n    return next(iter(found)) if len(found) == 1 else \'unknown\'\n\n\ndef paired_files(name, extension, directory_pairs, case_names):\n    path = PurePosixPath(name)\n    candidates = {str(path.with_suffix(extension))}\n    parts = list(path.parts)\n    for i, part in enumerate(parts[:-1]):\n        for before, after in directory_pairs:\n            if part.casefold() == before.casefold():\n                changed = parts.copy(); changed[i] = after\n                candidates.add(str(PurePosixPath(*changed).with_suffix(extension)))\n    return sorted({actual for candidate in candidates for actual in case_names.get(candidate.casefold(), [])})\n\n\ndef audit_dataset(reader, dataset, image_limit=None, hash_limit=2000, seed=42,\n                  verify_decode=False, progress=None, annotation_format=\'auto\'):\n    """All counts refer to scanned images; partial/hash coverage is explicit.\n\n    Zero objects means empty_annotation only for a recognized annotation file.\n    Missing/unreadable/ambiguous labels never become verified background images.\n    """\n    if annotation_format not in {\'auto\', \'yolo\', \'voc\', \'coco\'}:\n        raise ValueError(\'annotation format must be auto, yolo, voc or coco\')\n    rng = random.Random(seed)\n    images = sorted(name for name in reader.entries if PurePosixPath(name).suffix.lower() in IMAGE_EXTENSIONS)\n    if (image_limit is not None and image_limit < 0) or (hash_limit is not None and hash_limit < 0):\n        raise ValueError(\'Limits must be nonnegative or None\')\n    selected = images if image_limit is None else sorted(rng.sample(images, min(image_limit, len(images))))\n    hashed = set(selected if hash_limit is None else rng.sample(selected, min(hash_limit, len(selected))))\n    issues, image_rows, boxes, catalogs = [], [], [], []\n    case_names = defaultdict(list)\n    for name in reader.entries:\n        case_names[name.casefold()].append(name)\n\n    def issue(path, message):\n        issues.append({\'dataset\': dataset, \'path\': path, \'message\': str(message)[:600]})\n\n    for warning in reader.warnings:\n        issue(\'\', warning)\n    for name in reader.entries:\n        suffix = PurePosixPath(name).suffix.lower()\n        try:\n            names = None\n            if suffix in {\'.yaml\', \'.yml\'}:\n                config = yaml.safe_load(reader.read(name, 1024**2))\n                if isinstance(config, dict) and \'names\' in config:\n                    names = config[\'names\']\n            elif PurePosixPath(name).name.casefold() in {\'classes.txt\', \'obj.names\', \'classes.names\'}:\n                names = reader.read(name, 1024**2).decode(\'utf-8-sig\').splitlines()\n            if names is not None:\n                if isinstance(names, list):\n                    names = dict(enumerate(names))\n                if not isinstance(names, dict) or not names:\n                    raise ValueError(\'Invalid class names\')\n                normalized = {int(key): str(value).strip() for key, value in names.items()}\n                if any(key < 0 or not value for key, value in normalized.items()):\n                    raise ValueError(\'Invalid class ID/name\')\n                catalogs.append((name, normalized))\n        except Exception as error:\n            issue(name, f\'Class metadata: {type(error).__name__}: {error}\')\n\n    def catalog_for(image_name):\n        matches = [(name, names) for name, names in catalogs\n                   if str(PurePosixPath(name).parent) == \'.\'\n                   or image_name.startswith(str(PurePosixPath(name).parent) + \'/\')]\n        if matches:\n            depth = max(len(PurePosixPath(name).parent.parts) for name, _ in matches)\n            closest = [(name, names) for name, names in matches if len(PurePosixPath(name).parent.parts) == depth]\n            if len({json.dumps(names, sort_keys=True) for _, names in closest}) > 1:\n                issue(image_name, \'Conflicting class maps at the same directory; class IDs remain unmapped.\')\n                return (\'ambiguous-class-map\', {})\n            return closest[0]\n        unique = {json.dumps(names, sort_keys=True) for _, names in catalogs}\n        return catalogs[0] if len(unique) == 1 else (\'unmapped\', {})\n\n    # COCO indexing: resolve full/suffix names; ambiguous duplicate basenames are errors.\n    by_basename = defaultdict(list)\n    for name in images:\n        by_basename[PurePosixPath(name).name].append(name)\n    coco = defaultdict(list)\n    for name in reader.entries:\n        if annotation_format not in {\'auto\', \'coco\'} or PurePosixPath(name).suffix.lower() != \'.json\':\n            continue\n        try:\n            content = json.loads(reader.read(name))\n            if not isinstance(content, dict) or not {\'images\', \'annotations\', \'categories\'} <= content.keys():\n                issue(name, \'JSON is not COCO detection; inspect separately (e.g. LabelMe/classification).\')\n                continue\n            categories = {row[\'id\']: str(row[\'name\']) for row in content[\'categories\']}\n            annotations = defaultdict(list)\n            for annotation in content[\'annotations\']:\n                annotations[annotation[\'image_id\']].append(annotation)\n            for image in content[\'images\']:\n                filename = str(image[\'file_name\']).replace(\'\\\\\', \'/\')\n                candidates = by_basename[PurePosixPath(filename).name]\n                suffix_matches = [key for key in candidates if key == filename or key.endswith(\'/\' + filename)]\n                if suffix_matches:\n                    candidates = suffix_matches\n                # Do not join a JSON in one archive to same-named images in another.\n                container = name.split(\'!/\', 1)[0] if \'!/\' in name else None\n                candidates = [key for key in candidates if\n                              (key.startswith(container + \'!/\') if container else \'!/\' not in key)]\n                if len(candidates) != 1:\n                    issue(name, f\'COCO image reference missing/ambiguous: {filename}\')\n                    continue\n                coco[candidates[0]].append((name, categories, image, annotations[image[\'id\']]))\n        except Exception as error:\n            issue(name, f\'JSON: {type(error).__name__}: {error}\')\n\n    iterator = progress(selected, desc=dataset, unit=\'image\') if progress else selected\n    for name in iterator:\n        row = {\'dataset\': dataset, \'path\': name, \'split\': split_name(name), \'parent_folder\': str(PurePosixPath(name).parent),\n               \'width\': None, \'height\': None, \'mode\': None, \'format\': None,\n               \'annotation_status\': \'missing_annotation\', \'boxes\': 0, \'sha256\': None}\n        image_rows.append(row)\n        try:\n            with reader.open(name) as stream, Image.open(stream) as image:\n                row.update(width=image.width, height=image.height, mode=image.mode)\n                if verify_decode:\n                    image.load()\n            if name in hashed:\n                digest = hashlib.sha256()\n                with reader.open(name) as stream:\n                    for block in iter(lambda: stream.read(1024 * 1024), b\'\'):\n                        digest.update(block)\n                row[\'sha256\'] = digest.hexdigest()\n        except Exception as error:\n            row[\'annotation_status\'] = \'unreadable_image\'\n            issue(name, f\'Image: {type(error).__name__}: {error}\')\n            continue\n        txt = paired_files(name, \'.txt\', [(\'images\', \'labels\')], case_names)\n        xml = paired_files(name, \'.xml\', [(\'JPEGImages\', \'Annotations\'), (\'images\', \'annotations\')], case_names)\n        options = [(\'yolo\', path) for path in txt] + [(\'voc\', path) for path in xml] + [(\'coco\', entry) for entry in coco[name]]\n        if annotation_format != \'auto\':\n            options = [option for option in options if option[0] == annotation_format]\n        if not options:\n            continue\n        if len(options) > 1:\n            row[\'annotation_status\'] = \'ambiguous_annotation\'\n            issue(name, \'Multiple matching annotations/formats; select a single export/root before training.\')\n            continue\n        kind, annotation = options[0]\n        row[\'format\'] = kind\n        raw_boxes, bad = [], False\n        try:\n            if kind == \'yolo\':\n                source, names = catalog_for(name)\n                for line_number, line in enumerate(reader.read(annotation, 8 * 1024**2).decode(\'utf-8-sig\').splitlines(), 1):\n                    if not line.strip():\n                        continue\n                    try:\n                        fields = line.split()\n                        if len(fields) != 5:\n                            raise ValueError(\'YOLO detection expects 5 columns; segmentation/other format is not inferred\')\n                        class_id = int(fields[0])\n                        if class_id < 0 or names and class_id not in names:\n                            raise ValueError(\'Class ID outside class map\')\n                        cx, cy, width, height = map(float, fields[1:])\n                        raw_boxes.append((str(class_id), names.get(class_id, f\'class_{class_id}\'), bool(names),\n                                          source, cx - width/2, cy - height/2, width, height, 0))\n                    except Exception as error:\n                        bad = True; issue(annotation, f\'Line {line_number}: {error}\')\n            elif kind == \'voc\':\n                tree = ET.fromstring(reader.read(annotation, 8 * 1024**2))\n                if tree.tag != \'annotation\':\n                    raise ValueError(\'Not a VOC annotation\')\n                for dimension in (\'width\', \'height\'):\n                    value = tree.findtext(\'size/\' + dimension)\n                    if value is not None and int(value) != row[dimension]:\n                        raise ValueError(\'VOC annotation size differs from image\')\n                for obj in tree.findall(\'object\'):\n                    label = obj.findtext(\'name\')\n                    if not label:\n                        raise ValueError(\'VOC object without class name\')\n                    xmin, ymin, xmax, ymax = [float(obj.findtext(\'bndbox/\' + key)) for key in (\'xmin\', \'ymin\', \'xmax\', \'ymax\')]\n                    raw_boxes.append((label, label, True, \'voc:names\', xmin/row[\'width\'], ymin/row[\'height\'],\n                                      (xmax-xmin)/row[\'width\'], (ymax-ymin)/row[\'height\'], 0))\n            else:\n                annotation_path, categories, image, objects = annotation\n                for dimension in (\'width\', \'height\'):\n                    if dimension in image and int(image[dimension]) != row[dimension]:\n                        raise ValueError(\'COCO annotation size differs from image\')\n                for obj in objects:\n                    class_id = obj[\'category_id\']\n                    if class_id not in categories:\n                        raise ValueError(\'COCO category missing from categories\')\n                    x, y, width, height = map(float, obj[\'bbox\'])\n                    raw_boxes.append((str(class_id), categories[class_id], True, annotation_path,\n                                      x/row[\'width\'], y/row[\'height\'], width/row[\'width\'], height/row[\'height\'], int(obj.get(\'iscrowd\', 0))))\n            for class_id, label, mapped, schema, x, y, width, height, crowd in raw_boxes:\n                if not all(math.isfinite(v) for v in (x, y, width, height)) or min(width, height) <= 0 or min(x, y) < -1e-6 or max(x+width, y+height) > 1+1e-6:\n                    bad = True; issue(name, \'Invalid bbox: nonfinite, empty, negative, or outside image\'); continue\n                boxes.append({\'dataset\': dataset, \'path\': name, \'split\': row[\'split\'], \'format\': kind,\n                              \'label_namespace\': schema, \'class_id\': class_id, \'class_name\': label, \'class_mapped\': mapped,\n                              \'x_norm\': x, \'y_norm\': y, \'width_norm\': width, \'height_norm\': height,\n                              \'area_fraction\': width * height, \'iscrowd\': crowd,\n                              \'short_side_at_640\': min(width*row[\'width\'], height*row[\'height\']) * 640 / max(row[\'width\'], row[\'height\'])})\n                row[\'boxes\'] += 1\n            row[\'annotation_status\'] = \'invalid_annotation\' if bad else \'annotated\' if row[\'boxes\'] else \'empty_annotation\'\n        except Exception as error:\n            row[\'annotation_status\'] = \'invalid_annotation\'\n            issue(name, f\'{kind}: {type(error).__name__}: {error}\')\n\n    summary = {\'dataset\': dataset, \'root\': str(reader.root), \'images_total\': len(images), \'images_scanned\': len(image_rows),\n               \'partial_scan\': len(selected) < len(images), \'images_hashed\': sum(bool(row[\'sha256\']) for row in image_rows),\n               \'hash_partial\': sum(bool(row[\'sha256\']) for row in image_rows) < len(images),\n               \'verify_decode\': verify_decode, \'annotation_format\': annotation_format,\n               \'valid_boxes\': len(boxes), \'issues\': len(issues),\n               \'invalid_annotations\': sum(row[\'annotation_status\'] == \'invalid_annotation\' for row in image_rows),\n               \'annotation_status\': dict(Counter(row[\'annotation_status\'] for row in image_rows)),\n               \'formats\': dict(Counter(row[\'format\'] for row in image_rows if row[\'format\'])),\n               \'file_extensions\': dict(Counter(PurePosixPath(name).suffix.lower() for name in reader.entries)),\n               \'class_maps\': [{\'path\': name, \'names\': names} for name, names in catalogs]}\n    return {\'summary\': summary, \'images\': image_rows, \'boxes\': boxes, \'issues\': issues}\n\n\ndef duplicate_groups(image_rows):\n    grouped = defaultdict(list)\n    for row in image_rows:\n        if row.get(\'sha256\'):\n            grouped[row[\'sha256\']].append(row)\n    return [{\'sha256\': digest, \'copies\': len(rows), \'cross_dataset\': len({r[\'dataset\'] for r in rows}) > 1,\n             \'cross_split\': len({r[\'split\'] for r in rows} - {\'unknown\'}) > 1,\n             \'members\': [{\'dataset\': r[\'dataset\'], \'path\': r[\'path\'], \'split\': r[\'split\']} for r in rows]}\n            for digest, rows in grouped.items() if len(rows) > 1]\n', 'training/data/cache.py': '"""Standalone SupSign subset preparation and YOLO11s experiments (no API changes)."""\nfrom collections import Counter, defaultdict\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path, PurePosixPath\nimport random\nimport re\nimport shutil\nimport tempfile\nimport zipfile\n\nfrom PIL import Image\nimport yaml\n\nfrom training.data.audit import DatasetReader, audit_dataset\n\nfrom training.data.common import *\nfrom training.compatibility import require_compatible_recipe, dataset_content_fingerprint\n\ndef verify_dataset(root):\n    root = Path(root).resolve();manifest = read_json(root/\'manifest.json\')\n    require_compatible_recipe(manifest[\'recipe\'])\n    config = yaml.safe_load((root/\'data.yaml\').read_text(encoding=\'utf-8\'))\n    if config != dict(path=\'.\',train=\'images/train\',val=\'images/val\',test=\'images/test\',names=manifest[\'names\']):\n        raise ValueError(\'Dataset YAML changed\')\n    hashes,groups,expected,labels = set(),{},set(),set()\n    for row in manifest[\'images\']:\n        image,label = root/row[\'image\'],root/row[\'label\'];split=row[\'split\']\n        if split not in SPLITS or image.parent != root/\'images\'/split or label != root/\'labels\'/split/(image.stem+\'.txt\'):\n            raise ValueError(\'Unsafe or misplaced manifest path\')\n        if row[\'sha256\'] in hashes or (row[\'group\'] in groups and groups[row[\'group\']]!=split):\n            raise ValueError(\'Duplicate image or cross-split capture group\')\n        if digest_file(image)!=row[\'sha256\'] or digest_file(label)!=row[\'label_sha256\']:raise ValueError(\'Dataset image/label changed\')\n        hashes.add(row[\'sha256\']);groups[row[\'group\']]=split;expected.add(image);labels.add(label)\n    if expected!=set((root/\'images\').rglob(\'*.*\')) or labels!=set((root/\'labels\').rglob(\'*.txt\')):\n        raise ValueError(\'Dataset inventory changed\')\n    if {r[\'split\'] for r in manifest[\'images\']}!=set(SPLITS):raise ValueError(\'Empty dataset split\')\n    return dict(images=len(expected),names=manifest[\'names\'],manifest_sha256=digest_file(root/\'manifest.json\'),\n                dataset_content_sha256=dataset_content_fingerprint(manifest),\n                independent_field_test=manifest[\'independent_field_test\'],split_provenance=manifest[\'split_provenance\'])\n\n\ndef summarize_prepared_dataset(root, *, imgsz=640):\n    """EDA of the exact prepared training set, without changing images/labels/splits."""\n    root = Path(root).resolve()\n    if type(imgsz) is not int or imgsz < 32:\n        raise ValueError(\'Invalid EDA input image size\')\n    manifest = read_json(root / \'manifest.json\')\n    names = manifest[\'names\']\n    images, boxes = [], []\n    counts = {split: Counter() for split in SPLITS}\n    negatives = 0\n    for row in manifest[\'images\']:\n        # Preparation/cache verification already checked hashes and decoded images.\n        with Image.open(root / row[\'image\']) as image:\n            width, height = image.size\n        entries = [line.split() for line in (root / row[\'label\']).read_text().splitlines() if line.strip()]\n        if not entries:\n            negatives += 1\n        images.append(dict(image=row[\'image\'], split=row[\'split\'], group=row[\'group\'],\n                           width=width, height=height, boxes=len(entries)))\n        for entry in entries:\n            cls = int(entry[0]); _, _, w, h = map(float, entry[1:])\n            counts[row[\'split\']][cls] += 1\n            scale = imgsz / max(width, height)\n            boxes.append(dict(image=row[\'image\'], split=row[\'split\'], class_id=cls, name=names[cls],\n                              width_px=w*width, height_px=h*height, area_fraction=w*h,\n                              aspect_ratio=w*width/(h*height),\n                              short_side_at_input=min(w*width,h*height)*scale))\n    catalogue = [dict(class_id=i, name=name, **{s: counts[s][i] for s in SPLITS})\n                 for i, name in enumerate(names)]\n    exclusions = Counter(row[\'reason\'] for row in manifest[\'excluded\'])\n    summary = dict(images=len(images), boxes=len(boxes), negative_images=negatives,\n                   groups=len({r[\'group\'] for r in images}), input_size=imgsz,\n                   boxes_under_16px=sum(b[\'short_side_at_input\'] < 16 for b in boxes),\n                   duplicates_removed=manifest[\'duplicates_removed\'], excluded_images=len(manifest[\'excluded\']),\n                   exclusion_reasons=dict(exclusions), missing_classes=manifest[\'missing_classes\'],\n                   split_provenance=manifest[\'split_provenance\'], independent_field_test=False,\n                   source_summary=manifest[\'source_summary\'], limitations=manifest[\'limitations\'])\n    return dict(summary=summary, class_counts=catalogue, images=images, boxes=boxes)\n\n\ndef pack_dataset(root,target):\n    root,target=Path(root).resolve(),Path(target).resolve()\n    if target.is_relative_to(root):raise ValueError(\'Dataset cache must be outside the dataset\')\n    if target.exists() or target.with_suffix(target.suffix+\'.sha256\').exists():raise FileExistsError(target)\n    verify_dataset(root);target.parent.mkdir(parents=True,exist_ok=True)\n    temporary=target.with_suffix(target.suffix+\'.part\')\n    if temporary.exists():raise FileExistsError(temporary)\n    try:\n        with zipfile.ZipFile(temporary,\'x\',compression=zipfile.ZIP_STORED) as archive:\n            for path in sorted(root.rglob(\'*\')):\n                if path.is_file():archive.write(path,path.relative_to(root).as_posix())\n        temporary.rename(target)\n        target.with_suffix(target.suffix+\'.sha256\').write_text(digest_file(target),encoding=\'utf-8\')\n    finally:\n        if temporary.exists():temporary.unlink()\n    return target\n\n\ndef restore_dataset(source,output,*,expected_recipe=None):\n    source,output=Path(source),Path(output).resolve()\n    if output.exists():raise FileExistsError(output)\n    if digest_file(source)!=source.with_suffix(source.suffix+\'.sha256\').read_text().strip():\n        raise ValueError(\'Dataset cache checksum mismatch\')\n    output.parent.mkdir(parents=True,exist_ok=True)\n    with zipfile.ZipFile(source) as archive, tempfile.TemporaryDirectory(prefix=\'.restore-\',dir=output.parent) as temporary:\n        stage=Path(temporary)/\'dataset\';stage.mkdir();seen=set()\n        for member in archive.infolist():\n            relative=PurePosixPath(member.filename)\n            if (relative.is_absolute() or \'..\' in relative.parts or \'\\\\\' in member.filename\n                    or \':\' in member.filename or member.filename in seen\n                    or (member.external_attr>>16)&0o170000==0o120000):raise ValueError(\'Unsafe cache ZIP path\')\n            seen.add(member.filename)\n            if member.is_dir():continue\n            path=stage/relative;path.parent.mkdir(parents=True,exist_ok=True)\n            with archive.open(member) as stream,path.open(\'wb\') as target:shutil.copyfileobj(stream,target)\n        manifest=read_json(stage/\'manifest.json\')\n        require_compatible_recipe(manifest[\'recipe\'], expected_recipe)\n        verify_dataset(stage);stage.rename(output)\n    return manifest\n', 'training/data/common.py': "import hashlib, json\nfrom pathlib import Path\n\nCAR35 = [\n    'Speed_limit_40', 'Speed_limit_50', 'Speed_limit_60', 'Speed_limit_70', 'Speed_limit_80',\n    'No_entry', 'No_cars', 'No_cars_turn_left', 'No_cars_turn_right',\n    'No_turn_left', 'No_turn_right', 'No_U_turn', 'No_cars_U_turn',\n    'No_straight_ahead', 'No_turn_left_and_U_turn', 'No_turn_left_and_right',\n    'No_parking', 'No_stopping_and_parking', 'No_parking_odd_days', 'No_parking_even_days',\n    'Go_straight', 'Keep_right', 'Roundabout', 'One_way_road', 'Road_reserved_for_cars',\n    'Pedestrian_crossing', 'Children_crossing', 'Road_works', 'Slow_down',\n    'Dangerous_curve_left', 'Dangerous_curve_right',\n    'No_overtaking', 'End_no_overtaking', 'Give_way', 'End_all_prohibitions',\n]\nSPLITS = ('train', 'val', 'test')\nVERSION = 2\n\n\ndef digest_file(path):\n    with Path(path).open('rb') as stream:\n        digest = hashlib.sha256()\n        for block in iter(lambda: stream.read(1024*1024), b''): digest.update(block)\n        return digest.hexdigest()\n\n\ndef write_json(path, value):\n    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding='utf-8'))\n\n\ndef recipe(classes=CAR35, seed=42, block_size=100, near_threshold=4, groups=None):\n    from training.compatibility import implementation_digest\n    group_digest = hashlib.sha256(json.dumps(groups, sort_keys=True).encode()).hexdigest()\n    return dict(version=VERSION, classes=list(classes), seed=seed, block_size=block_size,\n        near_threshold=near_threshold, groups_sha256=group_digest,\n        implementation_sha256=implementation_digest(),\n        reader_sha256=hashlib.sha256(Path(__file__).with_name('audit.py').read_text(encoding='utf-8').encode()).hexdigest())\n", 'training/data/downloads.py': '"""Download a shared Drive ZIP to local Colab storage, without account cookies."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport re\nfrom urllib.parse import parse_qs, urlparse\nimport zipfile\n\n\ndef drive_file_id(source):\n    source = str(source).strip()\n    if not source or \'YOUR_\' in source or \'PASTE_\' in source:\n        raise ValueError(\'Paste a shared Google Drive file URL or file ID\')\n    if \'://\' in source:\n        parsed = urlparse(source)\n        if parsed.scheme != \'https\' or parsed.hostname != \'drive.google.com\' or parsed.username:\n            raise ValueError(\'Use a Google Drive file link, not a folder or another website\')\n        match = re.fullmatch(r\'/file/d/([A-Za-z0-9_-]+)(?:/.*)?\', parsed.path)\n        if match:\n            source = match[1]\n        elif parsed.path in (\'/uc\', \'/open\'):\n            source = parse_qs(parsed.query).get(\'id\', [\'\'])[0]\n        else:\n            raise ValueError(\'Use a Google Drive file link, not a folder\')\n    if not re.fullmatch(r\'[A-Za-z0-9_-]{10,}\', source):\n        raise ValueError(\'Invalid Google Drive file ID\')\n    return source\n\n\ndef download_drive_zip(source, target, *, expected_sha256=\'\', downloader=None):\n    """Publish only a complete ZIP; source receipts prevent resuming another file.\n\n    gdown owns its resumable temporary files. This helper keeps those on network\n    failure and never overwrites a pre-existing target from an unknown source.\n    An authoritative SHA-256 can adopt an already downloaded matching ZIP.\n    """\n    file_id = drive_file_id(source)\n    checksum = str(expected_sha256).strip().lower()\n    if checksum and not re.fullmatch(\'[a-f0-9]{64}\', checksum):\n        raise ValueError(\'Expected SHA-256 must have 64 hexadecimal characters\')\n    target = Path(target).resolve()\n    receipt = target.with_name(target.name+\'.source.json\')\n    staging = target.with_name(target.name+\'.download\')\n\n    def validate(path):\n        if not path.is_file() or not zipfile.is_zipfile(path):\n            raise ValueError(\'Download is not a ZIP. Check Drive sharing, quota and the file link.\')\n        if checksum:\n            with path.open(\'rb\') as stream:\n                actual = hashlib.file_digest(stream, \'sha256\').hexdigest()\n            if actual != checksum: raise ValueError(\'Downloaded ZIP SHA-256 mismatch\')\n\n    known = receipt.exists()\n    if known:\n        try:\n            same_source = json.loads(receipt.read_text())[\'file_id\'] == file_id\n        except (ValueError, KeyError, TypeError):\n            same_source = False\n        if not same_source: raise FileExistsError(\'Local download belongs to another Drive file; choose a new path\')\n    elif target.exists():\n        if not checksum: raise FileExistsError(\'Existing ZIP has no source receipt; supply its SHA-256 or choose a new path\')\n        validate(target)\n    elif staging.exists() or any(target.parent.glob(staging.name+\'*.tmp\')):\n        raise FileExistsError(\'Unknown partial download exists; choose a new local path\')\n    target.parent.mkdir(parents=True, exist_ok=True)\n    if not known:\n        with receipt.open(\'x\', encoding=\'utf-8\') as stream:\n            json.dump({\'file_id\': file_id}, stream)\n    if target.exists():\n        validate(target)\n        return target\n    if staging.exists():\n        try:\n            validate(staging)\n        except ValueError:\n            staging.unlink()  # Only the staging file identified by our matching receipt.\n    if not staging.exists():\n        if downloader is None:\n            import gdown\n            downloader = gdown.download\n        try:\n            result = downloader(id=file_id, output=str(staging), quiet=False,\n                                resume=True, use_cookies=False)\n        except Exception as error:\n            raise RuntimeError(\'Drive download failed. Check public-link access/quota, then rerun this cell \'\n                               \'to resume in the same runtime.\') from error\n        if result is None:\n            raise RuntimeError(\'Drive returned no download. Check public-link access and download quota.\')\n    try:\n        validate(staging)\n    except ValueError:\n        staging.unlink(missing_ok=True)\n        raise\n    if target.exists(): raise FileExistsError(\'Download target appeared; it was not overwritten\')\n    staging.rename(target)\n    return target\n', 'training/data/drive_source.py': '"""Resolve one user-supplied Drive ZIP. No extraction, training, or credentials."""\nfrom pathlib import Path\nimport re\nfrom time import time_ns\nimport zipfile\n\n\ndef validate_archive(path):\n    path = Path(path).resolve()\n    if not path.is_file():\n        raise FileNotFoundError(f\'Không thấy ZIP: {path}. Kiểm tra đường dẫn trên Drive.\')\n    try:\n        with zipfile.ZipFile(path) as archive:\n            if not archive.infolist():\n                raise ValueError(\'ZIP rỗng; chưa có file để EDA.\')\n            if any(item.flag_bits & 1 for item in archive.infolist()):\n                raise ValueError(\'ZIP có mật khẩu; cần bản ZIP không mã hóa để đọc ảnh/nhãn.\')\n    except zipfile.BadZipFile as error:\n        raise ValueError(\'File không phải ZIP hợp lệ; có thể là trang đăng nhập/quota Drive hoặc tải chưa đủ.\') from error\n    return path\n\n\ndef resolve_archive(file_id, cache_root, *, local_archive=None, download=None):\n    if not isinstance(file_id, str) or not re.fullmatch(r\'[A-Za-z0-9_-]+\', file_id):\n        raise ValueError(\'Drive file ID không hợp lệ; chỉ điền ID, không phải URL.\')\n    if local_archive:\n        return validate_archive(local_archive)\n    directory = Path(cache_root).resolve() / file_id\n    destination = directory / \'archive.zip\'\n    if destination.exists():\n        return validate_archive(destination)\n    directory.mkdir(parents=True, exist_ok=True)\n    if download is None:\n        import gdown\n        download = gdown.download\n    partial = directory / \'archive.zip.part\'\n    try:\n        result = download(id=file_id, output=str(partial), quiet=False, resume=True, use_cookies=False)\n    except Exception as error:\n        raise RuntimeError(\'Chưa tải được file Drive. Kiểm tra quyền xem/quota hoặc dùng SOURCE_MODE="drive" để đọc file gốc qua Drive mount.\') from error\n    if result is None or not partial.is_file():\n        raise RuntimeError(\'Drive chưa trả file. Dùng Drive mount nếu link cần đăng nhập hoặc đang vượt quota.\')\n    try:\n        validate_archive(partial)\n    except ValueError:\n        # Quarantine a completed HTML/broken response so resume will not reuse it.\n        partial.rename(directory / f\'archive.invalid-{time_ns()}.download\')\n        raise\n    # Publish only a complete ZIP. gdown owns its incomplete resume files.\n    if destination.exists():\n        raise FileExistsError(\'Cache đã xuất hiện trong lúc tải; không ghi đè.\')\n    partial.rename(destination)\n    return destination\n', 'training/data/supsign.py': '"""Standalone SupSign subset preparation and YOLO11s experiments (no API changes)."""\nfrom collections import Counter, defaultdict\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path, PurePosixPath\nimport random\nimport re\nimport shutil\nimport tempfile\nimport zipfile\n\nfrom PIL import Image\nimport yaml\n\nfrom training.data.audit import DatasetReader, audit_dataset\n\nfrom training.data.common import *\ndef visual_hash(image):\n    grey = image.convert(\'L\').resize((9, 8), Image.Resampling.LANCZOS)\n    pixels = list(grey.tobytes()); result = 0\n    for y in range(8):\n        for x in range(8): result = (result << 1) | (pixels[y*9+x] > pixels[y*9+x+1])\n    return result\n\n\ndef cluster_records(rows, *, groups=None, block_size=100, near_threshold=4, progress=None):\n    """Union supplied captures, filename blocks and dHash neighbours transitively.\n\n    Filename blocks / visual hashes are only development heuristics, never proof of\n    independent trips or physical signs. Explicit groups replace filename blocks.\n    """\n    if block_size < 1 or (near_threshold is not None and not 0 <= near_threshold <= 8):\n        raise ValueError(\'Invalid grouping settings\')\n    parent = list(range(len(rows)))\n    def find(i):\n        while i != parent[i]: parent[i] = parent[parent[i]]; i = parent[i]\n        return i\n    def union(a, b):\n        a, b = find(a), find(b)\n        if a != b: parent[max(a,b)] = min(a,b)\n    seen = {}\n    tree = None\n    grouping = progress(rows, desc=\'Group related images\', unit=\'image\') if progress else rows\n    for index, row in enumerate(grouping):\n        for alias in row[\'aliases\']:\n            relative = alias.split(\'!/\', 1)[-1]\n            if groups is not None:\n                group = groups.get(relative, groups.get(alias))\n                if not isinstance(group, str) or not group.strip():\n                    raise ValueError(f\'Missing capture group: {relative}\')\n                key = (\'capture\', group.strip())\n            else:\n                match = re.fullmatch(r\'(.*?)([0-9]+)\', PurePosixPath(relative).stem)\n                key = (\'filename\', match[1], int(match[2]) // block_size) if match else (\'image\', row[\'sha256\'])\n            if key in seen: union(index, seen[key])\n            else: seen[key] = index\n        if near_threshold is None: continue\n        value = row[\'visual_hash\']\n        if tree is None: tree = (value, index, {}); continue\n        # BK tree lookup: no O(n^2) all-pairs hash matrix.\n        pending = [tree]\n        while pending:\n            other, representative, children = pending.pop()\n            distance = (value ^ other).bit_count()\n            if distance <= near_threshold: union(index, representative)\n            pending.extend(child for edge, child in children.items()\n                           if distance-near_threshold <= edge <= distance+near_threshold)\n        node = tree\n        while True:\n            distance = (value ^ node[0]).bit_count()\n            if distance == 0: break\n            if distance not in node[2]: node[2][distance] = (value, index, {}); break\n            node = node[2][distance]\n    return [f\'g{find(i):06d}\' for i in range(len(rows))]\n\n\ndef choose_splits(rows, names, seed):\n    grouped = defaultdict(list)\n    for row in rows: grouped[row[\'group\']].append(row)\n    if len(grouped) < 3: raise ValueError(\'Need at least 3 independent candidate groups; provide capture groups\')\n    keys = sorted(grouped)\n    group_counts = {key: Counter(int(line.split()[0]) for row in values for line in row[\'lines\'])\n                    for key, values in grouped.items()}\n    total = sum(group_counts.values(), Counter())\n    absent = [names[i] for i in range(len(names)) if not total[i]]\n    if absent: raise ValueError(\'No usable training annotations for classes: \'+str(absent))\n    support = {i: {key for key in keys if group_counts[key][i]} for i in range(len(names))}\n    forced_train = {next(iter(groups)) for groups in support.values() if len(groups) == 1}\n    eligible = [key for key in keys if key not in forced_train]\n    # Removing two holdout groups loses coverage only for a class supported by\n    # exactly those two groups. This finds a feasible seed pair without splitting\n    # a group or depending on forty random attempts to rescue a rare class.\n    incompatible = {frozenset(groups) for groups in support.values() if len(groups) == 2}\n    ratios = {\'train\': .7, \'val\': .15, \'test\': .15}\n    def balance(sizes):\n        return sum(abs(sizes[s]/len(rows)-ratios[s]) for s in SPLITS)\n    best = None\n    for trial in range(40):\n        rng = random.Random(seed+trial)\n        candidates = eligible.copy(); rng.shuffle(candidates)\n        pair = next(((a,b) for i,a in enumerate(candidates) for b in candidates[i+1:]\n                     if frozenset((a,b)) not in incompatible), None)\n        if pair is None:\n            raise ValueError(\'Cannot preserve train class coverage with non-empty val/test groups; \'\n                             \'provide more independent capture groups, or explicitly revise the taxonomy\')\n        assignments = dict.fromkeys(keys, \'train\')\n        sizes = Counter(train=len(rows))\n        counts = {\'train\': total.copy(), \'val\': Counter(), \'test\': Counter()}\n        for key, split in zip(pair, (\'val\',\'test\')):\n            assignments[key] = split\n            sizes[\'train\'] -= len(grouped[key]); sizes[split] += len(grouped[key])\n            counts[\'train\'].subtract(group_counts[key]); counts[split].update(group_counts[key])\n        order = keys.copy(); rng.shuffle(order)\n        order.sort(key=lambda k: -len(grouped[k]))\n        missing_count = sum(not counts[s][i] for s in (\'val\',\'test\') for i in range(len(names)))\n        score = missing_count*1e6 + balance(sizes)\n        for key in order:\n            gc = group_counts[key]\n            if assignments[key] != \'train\' or any(counts[\'train\'][c] <= n for c,n in gc.items()):\n                continue\n            choices = []\n            for split in (\'val\',\'test\'):\n                proposed = sizes.copy()\n                proposed[\'train\'] -= len(grouped[key]); proposed[split] += len(grouped[key])\n                remaining = missing_count-sum(not counts[split][c] for c in gc)\n                choices.append((remaining*1e6+balance(proposed), split, remaining, proposed))\n            cost, split, remaining, proposed = min(choices, key=lambda v: v[0])\n            if cost < score:\n                assignments[key] = split; sizes = proposed; missing_count = remaining; score = cost\n                counts[\'train\'].subtract(gc); counts[split].update(gc)\n        missing = {s: [i for i in range(len(names)) if not counts[s][i]] for s in SPLITS}\n        if best is None or score < best[0]: best = score, assignments, counts, missing\n    _, assignments, counts, missing = best\n    if missing[\'train\']: raise ValueError(\'Missing train classes after grouping: \'+str([names[i] for i in missing[\'train\']]))\n    for row in rows: row[\'split\'] = assignments[row[\'group\']]\n    return counts, {s: [names[i] for i in missing[s]] for s in SPLITS}\n\n\ndef prepare_supsign(source, output, *, classes=CAR35, groups=None, seed=42,\n                    block_size=100, near_threshold=4, progress=None):\n    source, output = Path(source).resolve(), Path(output).resolve()\n    if not source.is_file() or not zipfile.is_zipfile(source): raise ValueError(\'SOURCE_ZIP must be an existing YOLO ZIP\')\n    if output.exists(): raise FileExistsError(output)\n    if not classes or len(set(classes)) != len(classes): raise ValueError(\'Choose unique target classes\')\n    settings = recipe(classes, seed, block_size, near_threshold, groups)\n    excluded, rows, by_hash = [], [], defaultdict(list)\n    with DatasetReader(source) as reader:\n        if reader.warnings: raise ValueError(\'Source archive warnings: \'+str(reader.warnings[:3]))\n        report = audit_dataset(reader, \'supsign\', hash_limit=None, verify_decode=True,\n                               annotation_format=\'yolo\', progress=progress)\n        source_names = {name for catalog in report[\'summary\'][\'class_maps\'] for name in catalog[\'names\'].values()}\n        missing = set(classes)-source_names\n        if missing: raise ValueError(\'Missing source classes: \'+str(sorted(missing)))\n        boxes = defaultdict(list)\n        for box in report[\'boxes\']: boxes[box[\'path\']].append(box)\n        for image in report[\'images\']:\n            status = image[\'annotation_status\']\n            if status not in (\'annotated\',\'empty_annotation\'):\n                excluded.append(dict(source_path=image[\'path\'],reason=status));continue\n            if any(not b[\'class_mapped\'] for b in boxes[image[\'path\']]):\n                raise ValueError(\'Unmapped source class metadata\')\n            by_hash[image[\'sha256\']].append(image)\n        duplicates_removed = 0\n        candidates = sorted(by_hash.items())\n        if progress: candidates = progress(candidates, desc=\'Deduplicate / visual hashes\', unit=\'image\')\n        for digest, copies in candidates:\n            # Compare ALL source labels, not just selected classes. Disagreement is quarantined.\n            signatures = {tuple(sorted((b[\'class_name\'], *(round(b[k],8) for k in\n                          [\'x_norm\',\'y_norm\',\'width_norm\',\'height_norm\'])) for b in boxes[v[\'path\']])) for v in copies}\n            if len(signatures) != 1:\n                excluded.extend(dict(source_path=v[\'path\'],reason=\'conflicting_duplicate_labels\') for v in copies);continue\n            representative = min(copies,key=lambda v:v[\'path\']); lines = []\n            for b in boxes[representative[\'path\']]:\n                if b[\'class_name\'] not in classes:continue\n                x,y = b[\'x_norm\']+b[\'width_norm\']/2,b[\'y_norm\']+b[\'height_norm\']/2\n                lines.append(f"{classes.index(b[\'class_name\'])} {x:.10f} {y:.10f} {b[\'width_norm\']:.10f} {b[\'height_norm\']:.10f}")\n            with reader.open(representative[\'path\']) as stream, Image.open(stream) as image:\n                value = visual_hash(image) if near_threshold is not None else 0\n            rows.append(dict(source_path=representative[\'path\'],aliases=sorted(v[\'path\'] for v in copies),\n                             sha256=digest,lines=lines,visual_hash=value))\n            duplicates_removed += len(copies)-1\n        if not rows: raise ValueError(\'No usable images remain\')\n        for row, group in zip(rows,cluster_records(rows,groups=groups,block_size=block_size,\n                                                 near_threshold=near_threshold,progress=progress)):\n            row[\'group\'] = group\n        report[\'_split_provenance\'] = \'provided_capture_groups\' if groups is not None else \'development_heuristic\'\n        return finish_supsign_preparation(source, output, rows=rows, report=report,\n            excluded=excluded, duplicates_removed=duplicates_removed, settings=settings, progress=progress)\n\n\ndef finish_supsign_preparation(source, output, *, rows, report, excluded,\n                               duplicates_removed, settings, progress=None):\n    """Finish an audited/grouped snapshot; verify copied image hashes before publishing."""\n    from copy import deepcopy\n    source, output = Path(source).resolve(), Path(output).resolve()\n    if output.exists(): raise FileExistsError(output)\n    rows = deepcopy(rows)  # Keep failed-frame state available if finalization needs retry.\n    classes, seed = settings[\'classes\'], settings[\'seed\']\n    counts, missing = choose_splits(rows,classes,seed)\n    with DatasetReader(source) as reader:\n        if reader.warnings: raise ValueError(\'Source archive warnings: \'+str(reader.warnings[:3]))\n        output.parent.mkdir(parents=True,exist_ok=True)\n        with tempfile.TemporaryDirectory(prefix=\'.supsign-\',dir=output.parent) as temporary:\n            stage = Path(temporary)/\'dataset\';stage.mkdir()\n            writing = progress(rows, desc=\'Write prepared images\', unit=\'image\') if progress else rows\n            for row in writing:\n                suffix = PurePosixPath(row[\'source_path\']).suffix.lower()\n                row[\'image\'] = f"images/{row[\'split\']}/{row[\'sha256\']}{suffix}"\n                row[\'label\'] = f"labels/{row[\'split\']}/{row[\'sha256\']}.txt"\n                image, label = stage/row[\'image\'], stage/row[\'label\']\n                image.parent.mkdir(parents=True,exist_ok=True);label.parent.mkdir(parents=True,exist_ok=True)\n                with reader.open(row[\'source_path\']) as stream, image.open(\'wb\') as target:shutil.copyfileobj(stream,target)\n                if digest_file(image) != row[\'sha256\']:raise ValueError(\'Source image changed during preparation\')\n                label.write_text(\'\\n\'.join(row.pop(\'lines\'))+\'\\n\',encoding=\'utf-8\')\n                row[\'label_sha256\'] = digest_file(label);row.pop(\'visual_hash\')\n            manifest = dict(kind=\'supsign_car35_experiment\',recipe=settings,names=list(classes),\n                source_zip_sha256=digest_file(source),source_summary=report[\'summary\'],\n                images=rows,excluded=excluded,issues=report[\'issues\'],duplicates_removed=duplicates_removed,\n                class_counts={s:dict(counts[s]) for s in SPLITS},missing_classes=missing,\n                split_provenance=report.get(\'_split_provenance\', \'development_heuristic\'),\n                independent_field_test=False,\n                limitations=\'Exact-byte duplicates removed; dHash and filename blocks are heuristics. \'\n                            \'An independent car/windshield field test is still required. \'\n                            \'Source-negative/other-class images are negatives only for the selected taxonomy.\')\n            write_json(stage/\'manifest.json\',manifest)\n            config=dict(path=\'.\',train=\'images/train\',val=\'images/val\',test=\'images/test\',names=list(classes))\n            (stage/\'data.yaml\').write_text(yaml.safe_dump(config,sort_keys=False),encoding=\'utf-8\')\n            stage.rename(output)\n    return manifest\n\n\ndef recover_failed_preparation(traceback, *, progress=None):\n    """Resume the split failure in the current Python session, without re-auditing.\n\n    The traceback must retain prepare_supsign\'s fully audited and grouped rows.\n    Never accepts an arbitrary saved file as proof that the source was audited.\n    """\n    frame = None\n    while traceback is not None:\n        candidate = traceback.tb_frame\n        if (candidate.f_code.co_name == \'prepare_supsign\' and\n            Path(candidate.f_code.co_filename).name in (\'supsign_training.py\', \'supsign.py\')):\n            state = candidate.f_locals\n            if (state.get(\'rows\') and all(\'group\' in row and \'lines\' in row for row in state[\'rows\'])\n                and all(key in state for key in (\'source\',\'output\',\'report\',\'excluded\',\'duplicates_removed\'))):\n                frame = dict(state)\n        traceback = traceback.tb_next\n    if frame is None:\n        raise ValueError(\'No completed preparation snapshot in traceback; keep the original failed runtime \'\n                         \'or rerun preparation with the updated notebook\')\n    def inventory(reader):\n        return {name: (member.CRC, member.file_size) for name, (_, member, _) in reader.entries.items()\n                if member is not None}\n    original = frame.get(\'reader\')\n    if original is None: raise ValueError(\'Missing source inventory in preparation traceback\')\n    with DatasetReader(frame[\'source\']) as current:\n        if inventory(original) != inventory(current):\n            raise ValueError(\'Source ZIP changed since the failed preparation; rerun the audit\')\n    report = dict(frame[\'report\'])\n    report[\'_split_provenance\'] = \'provided_capture_groups\' if frame.get(\'groups\') is not None else \'development_heuristic\'\n    settings = recipe(frame[\'classes\'], frame[\'seed\'], frame[\'block_size\'],\n                      frame[\'near_threshold\'], frame.get(\'groups\'))\n    return finish_supsign_preparation(frame[\'source\'], frame[\'output\'], rows=frame[\'rows\'],\n        report=report, excluded=frame[\'excluded\'], duplicates_removed=frame[\'duplicates_removed\'],\n        settings=settings, progress=progress)\n\n\n\nfrom training.data.cache import verify_dataset, summarize_prepared_dataset, pack_dataset, restore_dataset\nfrom training.experiments.train import train_experiment\nfrom training.experiments.evaluate import evaluate_checkpoint\n', 'training/experiments/__init__.py': '', 'training/experiments/evaluate.py': '"""Standalone SupSign subset preparation and YOLO11s experiments (no API changes)."""\nfrom collections import Counter, defaultdict\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path, PurePosixPath\nimport random\nimport re\nimport shutil\nimport tempfile\nimport zipfile\n\nfrom PIL import Image\nimport yaml\n\nfrom training.data.audit import DatasetReader, audit_dataset\n\nfrom training.data.common import *\nfrom training.data.cache import verify_dataset\n\ndef evaluate_checkpoint(checkpoint,root,output,*,split=\'val\',allow_test=False,imgsz=640,batch=8,yolo_factory=None):\n    checkpoint,root,output=Path(checkpoint),Path(root).resolve(),Path(output).resolve()\n    if split not in (\'val\',\'test\') or split==\'test\' and not allow_test:raise ValueError(\'Held-out test must be explicitly enabled after choosing settings\')\n    if output.exists():raise FileExistsError(output)\n    if output.is_relative_to(root):raise ValueError(\'Evaluation must be outside dataset\')\n    dataset=verify_dataset(root)\n    if yolo_factory is None:\n        from ultralytics import YOLO\n        yolo_factory=YOLO\n    detector=yolo_factory(str(checkpoint))\n    names=detector.names\n    names=[names[i] for i in range(len(names))] if isinstance(names,dict) else list(names)\n    if detector.task!=\'detect\' or names!=dataset[\'names\']:raise ValueError(\'Checkpoint taxonomy differs from dataset\')\n    output.mkdir(parents=True)\n    config=yaml.safe_load((root/\'data.yaml\').read_text());config[\'path\']=str(root)\n    runtime_data=output/\'data_runtime.yaml\';runtime_data.write_text(yaml.safe_dump(config),encoding=\'utf-8\')\n    result=detector.val(data=str(runtime_data),split=split,imgsz=imgsz,batch=batch,\n                        project=str(output.parent),name=output.name,exist_ok=True,plots=True)\n    report=dict(**dataset,split=split,checkpoint_sha256=digest_file(checkpoint),\n                metrics={k:float(v) if math.isfinite(float(v)) else None for k,v in result.results_dict.items()})\n    box=getattr(result,\'box\',None)\n    if box is not None:\n        report[\'per_class\']=[dict(class_id=int(i),name=names[int(i)],precision=float(box.p[j]),\n                                 recall=float(box.r[j]),map50=float(box.ap50[j]),map50_95=float(box.ap[j]))\n                             for j,i in enumerate(box.ap_class_index)]\n        report[\'classes_without_gt\']=[n for i,n in enumerate(names) if i not in set(box.ap_class_index)]\n    write_json(output/\'evaluation_report.json\',report)\n    return report\n', 'training/experiments/local_pipeline.py': '"""Local Vietnamese detection experiments. Never activates a checkpoint in the API.\n\nRun with: python -m scripts.local_detection_pipeline --help\n"""\nimport argparse\nfrom collections import Counter, defaultdict\nfrom contextlib import ExitStack\nimport csv\nfrom datetime import datetime, timezone\nimport hashlib\nimport json\nfrom pathlib import Path\nimport random\nimport shutil\nimport sys\nimport tempfile\n\nimport yaml\n\nfrom scripts.dataset_eda import DatasetReader, audit_dataset, duplicate_groups\n\nSPLITS = (\'train\', \'val\', \'test\')\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding=\'utf-8-sig\'))\n\n\ndef write_json(path, value):\n    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding=\'utf-8\')\n\n\ndef file_hash(path):\n    with Path(path).open(\'rb\') as stream:\n        return hashlib.file_digest(stream, \'sha256\').hexdigest()\n\n\ndef write_assignments(path, rows):\n    with Path(path).open(\'w\', newline=\'\', encoding=\'utf-8\') as stream:\n        writer = csv.DictWriter(stream, fieldnames=[\'dataset\', \'path\', \'split\', \'group\', \'exclude_reason\'])\n        writer.writeheader(); writer.writerows(rows)\n\n\ndef progress(items, *, desc, unit):\n    for index, item in enumerate(items, 1):\n        if index == 1 or index % 1000 == 0 or index == len(items):\n            print(f\'{desc}: {index}/{len(items)} {unit}s\', flush=True)\n        yield item\n\n\ndef inspect_sources(sources, output, annotation_format=\'auto\'):\n    """Full decode/hash audit, including ZIP members; no original file is changed."""\n    output = Path(output).resolve()\n    if output.exists():\n        raise FileExistsError(f\'Choose a new EDA directory: {output}\')\n    roots = {name: str(Path(root).resolve()) for name, root in sources.items()}\n    if not roots or any(not name for name in roots):\n        raise ValueError(\'Provide at least one named source\')\n    if any(output.is_relative_to(Path(root)) for root in roots.values()):\n        raise ValueError(\'EDA output must be outside every source directory\')\n    reports, mappings, assignments = {}, {}, []\n    for name, root in roots.items():\n        print(f\'Auditing {name}: decode + SHA-256 for all images...\', flush=True)\n        with DatasetReader(root) as reader:\n            report = audit_dataset(reader, name, hash_limit=None, verify_decode=True, progress=progress,\n                                   annotation_format=annotation_format)\n            report[\'archive_warnings\'] = reader.warnings\n        reports[name] = report\n        for box in report[\'boxes\']:\n            key = (name, box[\'label_namespace\'], box[\'class_id\'])\n            mappings[key] = {\'dataset\': name, \'namespace\': key[1], \'class_id\': key[2],\n                             \'source_name\': box[\'class_name\'],\n                             \'target\': box[\'class_name\'] if box[\'class_mapped\'] else None}\n        assignments.extend({\'dataset\': name, \'path\': image[\'path\'], \'split\': image[\'split\'],\n                            \'group\': \'\', \'exclude_reason\': \'\'} for image in report[\'images\'])\n        summary = report[\'summary\']\n        print(f\'{name}: {summary["images_scanned"]} images, {summary["valid_boxes"]} boxes, \'\n              f\'{summary["issues"]} issues; details in audit.json\', flush=True)\n    output.mkdir(parents=True)\n    write_json(output / \'audit.json\', reports)\n    write_json(output / \'duplicate_groups.json\', duplicate_groups(\n        [row for report in reports.values() for row in report[\'images\']]))\n    write_assignments(output / \'assignments.csv\', assignments)\n    write_json(output / \'plan.json\', {\n        \'sources\': roots, \'annotation_format\': annotation_format,\n        \'names\': sorted({item[\'target\'] for item in mappings.values() if item[\'target\']}),\n        \'class_map\': list(mappings.values()), \'assignments\': \'assignments.csv\',\n        \'reviewed\': False, \'split_reviewed\': False, \'empty_annotations_reviewed\': False,\n    })\n    print(f\'Review {output / "plan.json"} and assignments.csv before prepare.\', flush=True)\n\n\ndef split_groups(rows, seed=42):\n    """Explicit opt-in re-split, using user-supplied capture groups, never frames."""\n    included = [row for row in rows if not row.get(\'exclude_reason\', \'\').strip()]\n    if any(not row.get(\'group\', \'\').strip() for row in included):\n        raise ValueError(\'Fill a capture/location group for every included image before group split\')\n    groups = sorted({row[\'group\'].strip() for row in included})\n    if len(groups) < 3:\n        raise ValueError(\'Need at least 3 independent groups\')\n    random.Random(seed).shuffle(groups)\n    train = min(len(groups) - 2, max(1, int(len(groups) * .7)))\n    val = min(len(groups) - train - 1, max(1, int(len(groups) * .15)))\n    chosen = {group: \'train\' if i < train else \'val\' if i < train + val else \'test\'\n              for i, group in enumerate(groups)}\n    return [dict(row, split=chosen[row[\'group\'].strip()]) if not row.get(\'exclude_reason\', \'\').strip()\n            else dict(row) for row in rows]\n\n\ndef prepare_dataset(plan_path, output):\n    plan_path, output = Path(plan_path).resolve(), Path(output).resolve()\n    plan = read_json(plan_path)\n    if output.exists():\n        raise FileExistsError(f\'Output exists: {output}\')\n    if any(output.is_relative_to(Path(root).resolve()) for root in plan[\'sources\'].values()):\n        raise ValueError(\'Prepared output must be outside every source directory\')\n    if plan.get(\'reviewed\') is not True or plan.get(\'split_reviewed\') is not True:\n        raise ValueError(\'Set reviewed and split_reviewed only after inspecting labels and split provenance\')\n    names = plan[\'names\']\n    if not isinstance(names, list) or not names or any(not isinstance(n, str) or not n.strip() for n in names) or len(set(names)) != len(names):\n        raise ValueError(\'names must be a nonempty list of unique class names\')\n    class_map = {}\n    source_names = {}\n    for item in plan[\'class_map\']:\n        key = (item[\'dataset\'], item[\'namespace\'], str(item[\'class_id\']))\n        if key in class_map or item[\'target\'] not in names:\n            raise ValueError(f\'Missing/duplicate class mapping: {key}\')\n        class_map[key] = names.index(item[\'target\'])\n        source_names[key] = item[\'source_name\']\n    with (plan_path.parent / plan[\'assignments\']).open(newline=\'\', encoding=\'utf-8-sig\') as stream:\n        rows = list(csv.DictReader(stream))\n    assignments = {(row[\'dataset\'], row[\'path\']): row for row in rows}\n    if len(assignments) != len(rows):\n        raise ValueError(\'Duplicate assignment rows\')\n    output.parent.mkdir(parents=True, exist_ok=True)\n    # TemporaryDirectory owns only this newly created directory; originals are read-only.\n    with tempfile.TemporaryDirectory(prefix=\'.prepare-\', dir=output.parent) as temporary, ExitStack() as stack:\n        stage = Path(temporary) / \'dataset\'; stage.mkdir()\n        records, seen_hashes, seen_groups, observed = [], {}, {}, set()\n        class_counts = {split: Counter() for split in SPLITS}\n        excluded = []\n        for dataset, root in plan[\'sources\'].items():\n            reader = stack.enter_context(DatasetReader(root))\n            report = audit_dataset(reader, dataset, hash_limit=None, verify_decode=True, progress=progress,\n                                   annotation_format=plan.get(\'annotation_format\', \'auto\'))\n            if reader.warnings:\n                raise ValueError(f\'Unexpanded/broken archive in source {dataset}: {reader.warnings[:3]}\')\n            boxes = defaultdict(list)\n            for box in report[\'boxes\']:\n                boxes[box[\'path\']].append(box)\n            for image in report[\'images\']:\n                key = (dataset, image[\'path\']); observed.add(key)\n                if key not in assignments:\n                    raise ValueError(f\'New/unassigned image; rerun EDA: {key}\')\n                assignment = assignments[key]\n                if assignment.get(\'exclude_reason\', \'\').strip():\n                    excluded.append(assignment); continue\n                split, group = assignment[\'split\'], assignment.get(\'group\', \'\').strip()\n                if split not in SPLITS:\n                    raise ValueError(f\'Unknown split; fill assignments.csv or use group split: {key}\')\n                status = image[\'annotation_status\']\n                if status not in (\'annotated\', \'empty_annotation\'):\n                    raise ValueError(f\'{key}: {status}; fix or explicitly exclude this image\')\n                if status == \'empty_annotation\' and plan.get(\'empty_annotations_reviewed\') is not True:\n                    raise ValueError(f\'Review empty annotations as verified negatives first: {key}\')\n                digest = image[\'sha256\']\n                if digest in seen_hashes and seen_hashes[digest] != split:\n                    raise ValueError(f\'Identical image leaks across splits: {key}\')\n                seen_hashes[digest] = split\n                if group and group in seen_groups and seen_groups[group] != split:\n                    raise ValueError(f\'Capture group leaks across splits: {group}\')\n                if group:\n                    seen_groups[group] = split\n                lines = []\n                for box in boxes[image[\'path\']]:\n                    mapping = (dataset, box[\'label_namespace\'], box[\'class_id\'])\n                    if (mapping not in class_map or box[\'iscrowd\']\n                            or source_names[mapping] != box[\'class_name\']):\n                        raise ValueError(f\'Unmapped class or COCO crowd requires review: {key}, {mapping}\')\n                    class_id = class_map[mapping]\n                    x = box[\'x_norm\'] + box[\'width_norm\'] / 2\n                    y = box[\'y_norm\'] + box[\'height_norm\'] / 2\n                    lines.append(f\'{class_id} {x:.10f} {y:.10f} {box["width_norm"]:.10f} {box["height_norm"]:.10f}\')\n                    class_counts[split][class_id] += 1\n                filename = hashlib.sha256(json.dumps(key).encode()).hexdigest() + Path(image[\'path\']).suffix.lower()\n                relative = Path(\'images\') / split / filename\n                destination = stage / relative; destination.parent.mkdir(parents=True, exist_ok=True)\n                with reader.open(image[\'path\']) as source, destination.open(\'wb\') as target:\n                    shutil.copyfileobj(source, target, length=1024 * 1024)\n                if file_hash(destination) != digest:\n                    raise ValueError(f\'Source image changed during preparation: {key}\')\n                label_relative = Path(\'labels\') / split / Path(filename).with_suffix(\'.txt\')\n                label = stage / label_relative; label.parent.mkdir(parents=True, exist_ok=True)\n                label.write_text(\'\\n\'.join(lines) + (\'\\n\' if lines else \'\'), encoding=\'utf-8\')\n                records.append({\'source_dataset\': dataset, \'source_path\': image[\'path\'], \'split\': split,\n                                \'group\': group, \'image\': relative.as_posix(), \'sha256\': digest,\n                                \'label\': label_relative.as_posix(), \'label_sha256\': file_hash(label)})\n        if set(assignments) != observed:\n            raise ValueError(\'Stale assignments: source images removed or source selection changed; rerun EDA\')\n        if {record[\'split\'] for record in records} != set(SPLITS):\n            raise ValueError(\'Each of train, val and independent test must contain images\')\n        missing_train = [names[i] for i in range(len(names)) if not class_counts[\'train\'][i]]\n        if missing_train:\n            raise ValueError(f\'Every target class needs train annotations; missing: {missing_train}\')\n        config = {\'path\': str(output), **{split: f\'images/{split}\' for split in SPLITS}, \'names\': names}\n        (stage / \'data.yaml\').write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False), encoding=\'utf-8\')\n        manifest = {\'kind\': \'vietnam_local_experiment\', \'names\': names, \'class_map\': plan[\'class_map\'],\n                    \'annotation_format\': plan.get(\'annotation_format\', \'auto\'),\n                    \'sources\': plan[\'sources\'], \'images\': records, \'excluded\': excluded,\n                    \'class_counts\': class_counts,\n                    \'missing_classes\': {s: [names[i] for i in range(len(names)) if not class_counts[s][i]] for s in SPLITS},\n                    \'group_coverage\': sum(bool(r[\'group\']) for r in records),\n                    \'limitations\': \'Exact-byte duplicate check only. Review near-duplicate frames, split provenance, and missing classes.\'}\n        write_json(stage / \'manifest.json\', manifest)\n        # rename refuses to replace an existing directory on Windows.\n        stage.rename(output)\n    print(f\'Prepared {len(records)} images: {output / "data.yaml"}\', flush=True)\n    return manifest\n\n\ndef verify_prepared(data):\n    data = Path(data).resolve(); root = data.parent\n    config, manifest = yaml.safe_load(data.read_text(encoding=\'utf-8\')), read_json(root / \'manifest.json\')\n    if config.get(\'names\') != manifest[\'names\'] or Path(config.get(\'path\', \'\')).resolve() != root:\n        raise ValueError(\'Dataset path/names changed since preparation\')\n    if any(config.get(split) != f\'images/{split}\' for split in SPLITS):\n        raise ValueError(\'Split paths changed since preparation\')\n    expected = set()\n    seen_hashes, seen_groups = {}, {}\n    for record in manifest[\'images\']:\n        split = record[\'split\']\n        image, label = (root / record[key] for key in (\'image\', \'label\'))\n        if split not in SPLITS or not image.resolve().is_relative_to(root) or not label.resolve().is_relative_to(root):\n            raise ValueError(\'Unsafe manifest path or split\')\n        if image in expected or image.parent != root / \'images\' / split or label != root / \'labels\' / split / image.with_suffix(\'.txt\').name:\n            raise ValueError(\'Manifest contains repeated or misplaced image/label\')\n        expected.add(image)\n        if file_hash(image) != record[\'sha256\'] or file_hash(label) != record[\'label_sha256\']:\n            raise ValueError(f\'Prepared image/label changed; prepare a new version: {image}\')\n        for value, seen in ((record[\'sha256\'], seen_hashes), (record.get(\'group\'), seen_groups)):\n            if value and value in seen and seen[value] != split:\n                raise ValueError(\'Manifest contains cross-split content/group leakage\')\n            if value:\n                seen[value] = split\n    with DatasetReader(root) as reader:\n        audit = audit_dataset(reader, \'prepared\', hash_limit=0, verify_decode=True)\n    actual = {root / row[\'path\'] for row in audit[\'images\']}\n    if actual != expected or {r[\'split\'] for r in audit[\'images\']} != set(SPLITS):\n        raise ValueError(\'Prepared image inventory changed or a split is empty\')\n    if any(r[\'annotation_status\'] not in (\'annotated\', \'empty_annotation\') for r in audit[\'images\']):\n        raise ValueError(\'Invalid prepared labels/images\')\n    return {\'images\': len(expected), \'names\': manifest[\'names\'], \'manifest_sha256\': file_hash(root / \'manifest.json\'),\n            \'splits\': dict(Counter(r[\'split\'] for r in audit[\'images\'])),\n            \'missing_classes\': manifest[\'missing_classes\'], \'group_coverage\': manifest[\'group_coverage\']}\n\n\ndef run_model(command, data, model, output, *, allow_download=False, epochs=50, imgsz=640,\n              batch=2, device=\'0\', split=\'test\', yolo_factory=None):\n    output, data = Path(output).resolve(), Path(data).resolve()\n    if output.exists():\n        raise FileExistsError(f\'Choose a new run directory: {output}\')\n    if output.is_relative_to(data.parent):\n        raise ValueError(\'Run output must be outside the prepared dataset\')\n    if epochs < 1 or imgsz < 32 or batch < 1 or split not in (\'val\', \'test\'):\n        raise ValueError(\'Invalid training/evaluation settings\')\n    if not Path(model).is_file() and (command != \'train\' or not allow_download):\n        raise ValueError(\'Supply a local checkpoint, or allow official weight download with --allow-download\')\n    if not Path(model).is_file() and model not in (\'yolo11n.pt\', \'yolo11s.pt\', \'yolo11m.pt\', \'yolo11l.pt\', \'yolo11x.pt\'):\n        raise ValueError(\'Download supports official YOLO11 model names only\')\n    dataset = verify_prepared(data)\n    if yolo_factory is None:\n        from ai_engine.detection import configure_ultralytics\n        configure_ultralytics()\n        from ultralytics import YOLO\n        yolo_factory = YOLO\n    detector = yolo_factory(str(model), task=\'detect\')\n    options = dict(data=str(data), imgsz=imgsz, batch=batch, device=device, workers=0,\n                   project=str(output.parent), name=output.name, exist_ok=False, plots=True)\n    metrics = None\n    if command == \'train\':\n        detector.train(**options, epochs=epochs, seed=42, deterministic=True, cache=False,\n                       amp=False, fliplr=0., flipud=0., degrees=5.)\n        checkpoint = Path(detector.trainer.best)\n    elif command == \'evaluate\':\n        if detector.task != \'detect\' or [detector.names[i] for i in range(len(detector.names))] != dataset[\'names\']:\n            raise ValueError(\'Checkpoint taxonomy/task does not match the prepared dataset\')\n        result = detector.val(**options, split=split)\n        metrics = {key: float(value) for key, value in result.results_dict.items()}\n        checkpoint = Path(model).resolve()\n    else:\n        raise ValueError(f\'Unknown model command: {command}\')\n    output.mkdir(parents=True, exist_ok=True)\n    write_json(output / \'pipeline_report.json\', {\n        \'created_at\': datetime.now(timezone.utc).isoformat(), \'command\': command, **dataset,\n        \'checkpoint\': str(checkpoint.resolve()), \'checkpoint_sha256\': file_hash(checkpoint),\n        \'settings\': {**options, \'epochs\': epochs if command == \'train\' else None,\n                     **({\'seed\': 42, \'deterministic\': True, \'cache\': False, \'amp\': False,\n                         \'fliplr\': 0., \'flipud\': 0., \'degrees\': 5.} if command == \'train\' else {\'split\': split})},\n        \'test_evaluated\': command == \'evaluate\' and split == \'test\', \'metrics\': metrics,\n        \'limitation\': \'Experimental taxonomy; API integration requires explicit mapping and a compatible detector.\',\n    })\n    print(f\'Results: {output}\', flush=True)\n\n\ndef main():\n    # Windows redirected terminals may otherwise use cp1252, breaking Vietnamese class names.\n    for stream in (sys.stdout, sys.stderr):\n        if hasattr(stream, \'reconfigure\'):\n            stream.reconfigure(encoding=\'utf-8\')\n    parser = argparse.ArgumentParser(description=__doc__)\n    commands = parser.add_subparsers(dest=\'command\', required=True)\n    eda = commands.add_parser(\'eda\', help=\'Full local/ZIP audit and review-plan draft\')\n    eda.add_argument(\'--source\', action=\'append\', required=True, metavar=\'NAME=FOLDER\')\n    eda.add_argument(\'--output\', type=Path, required=True)\n    eda.add_argument(\'--annotation-format\', choices=(\'auto\', \'yolo\', \'voc\', \'coco\'), default=\'auto\')\n    prepare = commands.add_parser(\'prepare\')\n    prepare.add_argument(\'--plan\', type=Path, required=True)\n    prepare.add_argument(\'--output\', type=Path, required=True)\n    splitter = commands.add_parser(\'split\', help=\'Optional group re-split; writes a new CSV\')\n    splitter.add_argument(\'--assignments\', type=Path, required=True)\n    splitter.add_argument(\'--output\', type=Path, required=True)\n    splitter.add_argument(\'--seed\', type=int, default=42)\n    validate = commands.add_parser(\'validate\')\n    validate.add_argument(\'--data\', type=Path, required=True)\n    for command in (\'train\', \'evaluate\'):\n        runner = commands.add_parser(command)\n        runner.add_argument(\'--data\', type=Path, required=True)\n        runner.add_argument(\'--model\', required=True)\n        runner.add_argument(\'--output\', type=Path, required=True)\n        runner.add_argument(\'--allow-download\', action=\'store_true\')\n        runner.add_argument(\'--epochs\', type=int, default=50)\n        runner.add_argument(\'--imgsz\', type=int, default=640)\n        runner.add_argument(\'--batch\', type=int, default=2)\n        runner.add_argument(\'--device\', default=\'0\')\n        runner.add_argument(\'--split\', choices=(\'val\', \'test\'), default=\'test\')\n    args = parser.parse_args()\n    try:\n        if args.command == \'eda\':\n            pairs = [source.split(\'=\', 1) for source in args.source]\n            if any(len(pair) != 2 for pair in pairs) or len({pair[0] for pair in pairs}) != len(pairs):\n                raise ValueError(\'Use unique --source NAME=FOLDER entries\')\n            inspect_sources(dict(pairs), args.output, args.annotation_format)\n        elif args.command == \'prepare\':\n            prepare_dataset(args.plan, args.output)\n        elif args.command == \'split\':\n            if args.output.exists() or args.output.resolve() == args.assignments.resolve():\n                raise FileExistsError(\'Choose a new assignments output file\')\n            with args.assignments.open(newline=\'\', encoding=\'utf-8-sig\') as stream:\n                rows = split_groups(list(csv.DictReader(stream)), args.seed)\n            write_assignments(args.output, rows)\n        elif args.command == \'validate\':\n            print(json.dumps(verify_prepared(args.data), indent=2, ensure_ascii=False))\n        else:\n            run_model(**vars(args))\n    except (ValueError, OSError, KeyError, TypeError) as error:\n        parser.exit(1, f\'Pipeline stopped: {error}\\n\')\n\n\nif __name__ == \'__main__\':\n    main()\n', 'training/experiments/train.py': '"""Standalone SupSign subset preparation and YOLO11s experiments (no API changes)."""\nfrom collections import Counter, defaultdict\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path, PurePosixPath\nimport random\nimport re\nimport shutil\nimport tempfile\nimport zipfile\n\nfrom PIL import Image\nimport yaml\n\nfrom training.data.audit import DatasetReader, audit_dataset\n\nfrom training.data.common import *\nfrom training.data.cache import verify_dataset\n\ndef train_experiment(root,run_dir,*,epochs=80,imgsz=640,batch=8,device=\'0\',resume=False,yolo_factory=None):\n    root,run_dir=Path(root).resolve(),Path(run_dir).resolve()\n    if run_dir.is_relative_to(root):raise ValueError(\'Training run must be outside dataset\')\n    if epochs<1 or imgsz<32 or batch<1:raise ValueError(\'Invalid training settings\')\n    dataset=verify_dataset(root)\n    request_path=run_dir/\'training_request.json\'\n    settings=dict(model=\'yolo11s.pt\',epochs=epochs,imgsz=imgsz,batch=batch)\n    if resume:\n        if not (run_dir/\'weights/last.pt\').is_file() or not request_path.is_file():raise ValueError(\'Resume requires saved last.pt and training_request.json\')\n        previous=read_json(request_path)\n        if previous[\'manifest_sha256\']!=dataset[\'manifest_sha256\']:raise ValueError(\'Resume dataset changed\')\n        if previous[\'settings\']!=settings:raise ValueError(\'Resume settings changed; use the original settings or a new run\')\n        model=run_dir/\'weights/last.pt\'\n    else:\n        if run_dir.exists():raise FileExistsError(\'Run exists; set RESUME=True or choose another RUN_NAME\')\n        model=\'yolo11s.pt\'\n    if yolo_factory is None:\n        from ultralytics import YOLO\n        yolo_factory=YOLO\n    detector=yolo_factory(str(model))\n    if not resume:\n        run_dir.mkdir(parents=True)\n        write_json(request_path,dict(**dataset,settings=settings,test_evaluated=False))\n    # Dataset archive remains portable; runtime absolute path lives beside the run.\n    config=yaml.safe_load((root/\'data.yaml\').read_text(encoding=\'utf-8\'));config[\'path\']=str(root)\n    runtime_data=run_dir/\'data_runtime.yaml\';runtime_data.write_text(yaml.safe_dump(config,sort_keys=False),encoding=\'utf-8\')\n    options=dict(data=str(runtime_data),epochs=epochs,imgsz=imgsz,batch=batch,device=device,\n                 project=str(run_dir.parent),name=run_dir.name,exist_ok=True,workers=2,\n                 seed=42,deterministic=True,cache=False,amp=True,patience=20,save=True,save_period=5,\n                 fliplr=0.,flipud=0.,degrees=0.,shear=0.,perspective=0.,mixup=0.,\n                 mosaic=.5,close_mosaic=min(10,epochs),plots=True)\n    if resume:options[\'resume\']=True\n    detector.train(**options)\n    if not (run_dir/\'weights/best.pt\').is_file():raise ValueError(\'Training did not produce best.pt\')\n    return run_dir/\'weights/best.pt\'\n'}
CODE_ROOT = WORK / 'pipeline'
for relative, source in TRAINING_FILES.items():
    target = CODE_ROOT / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding='utf-8')
(CODE_ROOT / 'scripts/__init__.py').touch()
if str(CODE_ROOT) not in sys.path: sys.path.insert(0, str(CODE_ROOT))
from scripts.supsign_training import (CAR35, recipe, prepare_supsign, verify_dataset,
    digest_file, pack_dataset, restore_dataset, train_experiment, evaluate_checkpoint,
    summarize_prepared_dataset, recover_failed_preparation)
from scripts.colab_downloads import download_drive_zip
from training.compatibility import require_compatible_recipe
GROUPS = None
if GROUPS_CSV is not None:
    with Path(GROUPS_CSV).open(newline='', encoding='utf-8-sig') as stream:
        group_rows = list(csv.DictReader(stream))
    assert all(row.get('path') and row.get('group') for row in group_rows), 'CSV cần path,group.'
    GROUPS = {row['path']: row['group'] for row in group_rows}
    assert len(GROUPS) == len(group_rows), 'CSV có đường dẫn trùng.'
RECIPE = recipe(CAR35, SEED, FRAME_BLOCK_SIZE, NEAR_HASH_DISTANCE, GROUPS)
print('Taxonomy:', len(CAR35), 'lớp. Cache sẽ kiểm tra cùng recipe/code và checksum.')

## 2. Chuẩn bị dataset hoặc khôi phục cache

Lần đầu gdown tải ZIP về local, sau đó đọc/hash toàn bộ ZIP. Cần đủ ổ local cho ZIP và dataset/cache,
và đủ Drive nếu bật lưu Drive. Không giải nén ZIP nguồn tùy ý; chỉ ghi ảnh/nhãn đã kiểm tra.
Dataset local/cache sẵn có được ưu tiên; khi đổi tài khoản, PREPARED_CACHE_URL có thể dùng lại cache cũ.
Link Drive bị chặn/quota không được thay bằng dữ liệu giả. Tải gián đoạn có thể retry trong cùng runtime.
Cache có checksum; thay code/classes/grouping thì phải dùng một DATASET_VERSION mới.
V3 giữ đủ lớp trong train nếu có thể tạo ba split không rỗng mà không tách nhóm.
Lớp chỉ có trong một nhóm được giữ ở train; val/test có thể thiếu lớp đó và được ghi rõ.

Nhãn lỗi bị cách ly cả ảnh, không lặng lẽ sửa annotation nguồn. Ảnh trùng byte nhưng nhãn khác
cũng bị cách ly để xem lại. Ảnh chỉ chứa lớp ngoài scope được giữ làm nền đối với 35 lớp mục tiêu.

In [ ]:
from tqdm.auto import tqdm
if DATASET_ROOT.exists():
    info = verify_dataset(DATASET_ROOT)
    manifest = json.loads((DATASET_ROOT / 'manifest.json').read_text())
    require_compatible_recipe(manifest['recipe'], RECIPE)
    print('Dùng dataset local đã kiểm tra:', info['images'], 'ảnh')
elif CACHE_ZIP.is_file():
    manifest = restore_dataset(CACHE_ZIP, DATASET_ROOT, expected_recipe=RECIPE)
    print('Đã khôi phục cache, không đọc lại ZIP thô.')
elif PREPARED_CACHE_URL.strip():
    assert PREPARED_CACHE_SHA256.strip(), 'Điền SHA-256 từ file cache .zip.sha256.'
    shared_cache = download_drive_zip(PREPARED_CACHE_URL,
        WORK / 'downloads' / (DATASET_VERSION + '.zip'), expected_sha256=PREPARED_CACHE_SHA256)
    shared_cache.with_suffix(shared_cache.suffix + '.sha256').write_text(PREPARED_CACHE_SHA256.strip().lower())
    manifest = restore_dataset(shared_cache, DATASET_ROOT, expected_recipe=RECIPE)
    print('Đã khôi phục cache chia sẻ từ tài khoản khác; không audit ZIP gốc.')
else:
    if SOURCE_URL.strip():
        SOURCE_ZIP = download_drive_zip(SOURCE_URL, SOURCE_ZIP, expected_sha256=SOURCE_SHA256)
    assert SOURCE_ZIP.is_file(), 'Dán link ZIP/file ID vào SOURCE_URL ở cell Cấu hình.'
    assert zipfile.is_zipfile(SOURCE_ZIP), 'Nguồn tải về không phải ZIP hợp lệ.'
    with zipfile.ZipFile(SOURCE_ZIP) as source_archive:
        required = sum(member.file_size for member in source_archive.infolist()) * 2 + 2*1024**3
    free = shutil.disk_usage(WORK).free
    assert free >= required, f'Ổ local cần khoảng {required/1024**3:.1f} GiB trống; còn {free/1024**3:.1f}.'
    manifest = prepare_supsign(SOURCE_ZIP, DATASET_ROOT, classes=CAR35, groups=GROUPS,
        seed=SEED, block_size=FRAME_BLOCK_SIZE, near_threshold=NEAR_HASH_DISTANCE,
        progress=lambda items, desc, unit: tqdm(items, desc=desc, unit=unit))
    print('Đang kiểm tra và lưu cache dataset...', flush=True)
    pack_dataset(DATASET_ROOT, CACHE_ZIP)
    print('Đã lưu cache:', CACHE_ZIP)
from collections import Counter
split_counts = Counter(row['split'] for row in manifest['images'])
print(' | '.join(f'{s}: {split_counts[s]:,} ảnh' for s in ('train', 'val', 'test')))
print('Loại bản sao:', manifest['duplicates_removed'], '| Cách ly:', len(manifest['excluded']))
print('Split:', manifest['split_provenance'], '| Test thực địa độc lập:', manifest['independent_field_test'])
for split, missing in manifest['missing_classes'].items():
    if missing: print('Lớp thiếu ở', split, ':', ', '.join(missing))

## EDA trước khi train — lưu báo cáo trên Drive

Thống kê **đúng dataset 35 lớp sau khi lọc**, kèm audit nguồn trong `source_summary`/manifest.
Xuất JSON + CSV (ảnh, bbox, số mẫu từng lớp), biểu đồ phân bố lớp/split, kích thước ảnh,
diện tích và cạnh ngắn bbox ở kích thước train. `boxes_under_16px` giúp nhận ra biển nhỏ khó học.
Không audit lại ZIP thô nếu đã có cache. Chỉ in tóm tắt; chi tiết lỗi/cách ly nằm trong manifest.
Nhóm filename/dHash vẫn là heuristic; EDA không thay tập kiểm tra qua kính ô tô độc lập.

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter
eda = summarize_prepared_dataset(DATASET_ROOT, imgsz=IMAGE_SIZE)
EDA_ROOT = PERSIST / 'eda' / DATASET_VERSION
EDA_ROOT.mkdir(parents=True, exist_ok=True)
(EDA_ROOT / 'dataset_eda.json').write_text(json.dumps(eda['summary'], ensure_ascii=False, indent=2), encoding='utf-8')
for key in ('class_counts', 'images', 'boxes'):
    rows = eda[key]
    with (EDA_ROOT / (key + '.csv')).open('w', newline='', encoding='utf-8-sig') as stream:
        if rows:
            writer = csv.DictWriter(stream, fieldnames=list(rows[0])); writer.writeheader(); writer.writerows(rows)
fig, axes = plt.subplots(2, 2, figsize=(18, 14))
y = list(range(len(eda['class_counts']))); starts = [0] * len(y)
for split in ('train', 'val', 'test'):
    values = [r[split] for r in eda['class_counts']]
    axes[0,0].barh(y, values, left=starts, label=split)
    starts = [a+b for a,b in zip(starts, values)]
axes[0,0].set_yticks(y, [r['name'] for r in eda['class_counts']], fontsize=8)
axes[0,0].set_title('Boxes per class / split'); axes[0,0].legend()
axes[0,1].hist([r['short_side_at_input'] for r in eda['boxes']], bins=40)
axes[0,1].axvline(16, color='red', linestyle='--'); axes[0,1].set_title(f'BBox short side at input {IMAGE_SIZE} (px)')
axes[1,0].scatter([r['width'] for r in eda['images']], [r['height'] for r in eda['images']], s=3, alpha=.15)
axes[1,0].set(xlabel='Image width', ylabel='Image height', title='Image dimensions')
axes[1,1].hist([r['area_fraction'] for r in eda['boxes']], bins=40)
axes[1,1].set_title('BBox area / image area')
fig.tight_layout(); fig.savefig(EDA_ROOT / 'eda_overview.png', dpi=150); plt.show(); plt.close(fig)
summary = eda['summary']
print(f"EDA: {summary['images']:,} images | {summary['boxes']:,} boxes | {summary['negative_images']:,} negatives")
print(f"Small boxes (<16 px at {IMAGE_SIZE}): {summary['boxes_under_16px']:,}")
print('Duplicate copies removed:', summary['duplicates_removed'], '| Quarantined:', summary['excluded_images'])
print('Saved:', EDA_ROOT)

## 3. Kiểm tra nhãn và phân bố lớp

Xem ảnh đầy đủ, bbox và tên lớp trước khi train. Phân biệt cấm ô tô/cấm chung, cấm đỗ/cấm dừng,
biển hướng trái/phải. Class ID mới là 0–34 theo CAR35; **không dùng ID này thay trực tiếp vào database 10 lớp**.
Các lớp thiếu val/test phải được ghi nhận khi đọc metric; chưa có nhãn không có nghĩa model đạt chất lượng.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import random
sample_rows = manifest['images'].copy()
random.Random(SEED).shuffle(sample_rows)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for axis in axes.flat: axis.axis('off')
for axis, row in zip(axes.flat, sample_rows[:6]):
    with Image.open(DATASET_ROOT / row['image']) as source: image = source.convert('RGB')
    draw = ImageDraw.Draw(image)
    for line in (DATASET_ROOT / row['label']).read_text().splitlines():
        if not line.strip(): continue
        cls, x, y, w, h = map(float, line.split())
        box = ((x-w/2)*image.width, (y-h/2)*image.height, (x+w/2)*image.width, (y+h/2)*image.height)
        draw.rectangle(box, outline='yellow', width=2)
        draw.text((max(0, box[0]), max(0, box[1]-12)), CAR35[int(cls)], fill='yellow')
    axis.imshow(image); axis.set_title(row['split'])
plt.tight_layout(); fig.savefig(EDA_ROOT / 'bbox_preview.png', dpi=150); plt.show(); plt.close(fig)
# Gọn 35 dòng để thấy lớp ít mẫu; không in hàng chục nghìn đường dẫn.
print(f"{'Class':35s} {'train':>7s} {'val':>7s} {'test':>7s}")
for i, name in enumerate(CAR35):
    values = [manifest['class_counts'][s].get(str(i), manifest['class_counts'][s].get(i, 0)) for s in ('train', 'val', 'test')]
    print(f'{name:35s} {values[0]:7d} {values[1]:7d} {values[2]:7d}')

## 4. Tải báo cáo để xem trước

Gửi **EDA_REPORT_ZIP** tải xuống ở cell cuối, không cần gửi cache ZIP lớn.
Báo cáo gồm JSON/CSV, biểu đồ, ảnh bbox preview, manifest, audit nguồn và training_handoff.json.
Giữ cache ZIP + .sha256 trên Drive. Nếu dùng tài khoản khác, chia sẻ cache ZIP cho tải qua link;
copy cache_sha256 và manifest_sha256 trong training_handoff.json sang notebook train.
Nếu muốn đổi lớp sau khi xem EDA, cần chuẩn bị cache phiên bản mới; chưa train ở notebook này.

In [ ]:
from datetime import datetime, timezone
from google.colab import files
# Dataset local đã có nhưng chưa lưu cache (ví dụ runtime trước bị ngắt lúc pack).
cache_to_share = CACHE_ZIP if CACHE_ZIP.is_file() else globals().get('shared_cache')
if cache_to_share is None:
    pack_dataset(DATASET_ROOT, CACHE_ZIP)
    cache_to_share = CACHE_ZIP
handoff = dict(schema_version=1, dataset_version=DATASET_VERSION,
    cache_name=cache_to_share.name, cache_sha256=digest_file(cache_to_share),
    manifest_sha256=digest_file(DATASET_ROOT / 'manifest.json'),
    names=manifest['names'], recipe=manifest['recipe'], input_size=IMAGE_SIZE)
(EDA_ROOT / 'training_handoff.json').write_text(json.dumps(handoff, ensure_ascii=False, indent=2), encoding='utf-8')
(EDA_ROOT / 'source_summary.json').write_text(json.dumps(manifest['source_summary'], ensure_ascii=False, indent=2), encoding='utf-8')
for name in ('manifest.json', 'data.yaml'): shutil.copy2(DATASET_ROOT / name, EDA_ROOT / name)
exports = PERSIST / 'exports'; exports.mkdir(exist_ok=True)
report_name = DATASET_VERSION + '_eda_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
EDA_REPORT_ZIP = Path(shutil.make_archive(str(exports / report_name), 'zip', EDA_ROOT))
print('EDA report:', EDA_REPORT_ZIP)
print('Cache để train:', cache_to_share)
print('PREPARED_CACHE_SHA256 =', repr(handoff['cache_sha256']))
print('EXPECTED_MANIFEST_SHA256 =', repr(handoff['manifest_sha256']))
files.download(str(EDA_REPORT_ZIP))